# 04 - Transfer Learning
Train frozen ResNet50 and fine-tune the last layers.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import setup_logging, set_seed, Trainer
from src.data_loader import DataLoader
from src.model import ModelBuilder
from src.evaluate import Evaluator
from src import config
import tensorflow as tf

setup_logging(); set_seed()
loader = DataLoader()
train_gen, val_gen, test_gen = loader.get_generators()
class_weights = loader.get_class_weights(train_gen)
trainer = Trainer()
builder = ModelBuilder()
model = builder.build_transfer_model(base="resnet50")

In [ ]:
frozen_history = trainer.train(model, train_gen, val_gen, class_weights, "resnet50_frozen_notebook", config.EPOCHS, config.RESNET_FROZEN_MODEL_PATH)
trainer.plot_training_curves(frozen_history, "resnet50_frozen_notebook")

In [ ]:
model = builder.unfreeze_and_finetune(model, num_layers=20)
ft_history = trainer.train(model, train_gen, val_gen, class_weights, "resnet50_finetuned_notebook", config.EPOCHS + 15, config.RESNET_FINETUNE_MODEL_PATH, initial_epoch=config.EPOCHS)
trainer.plot_training_curves(ft_history, "resnet50_finetuned_notebook")

In [ ]:
model = tf.keras.models.load_model(str(config.RESNET_FINETUNE_MODEL_PATH))
evalr = Evaluator()
metrics = evalr.evaluate_model(model, test_gen, model_name="resnet50_finetuned_notebook")
metrics

## Transfer Learning Summary
- Frozen training adapts only the classifier head.
- Fine-tuning unfreezes last 20 backbone layers for domain adaptation.
- Compare metrics before/after fine-tuning to validate gains.